In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("fraud_detection.silver.transactions_enriched")

# Weights are explicitly justified below - not arbitrary
gold_risk_scored = (
    silver_df
    .withColumn(
        "risk_score",
        # price_out_of_range: the one statistically validated signal (7.25% vs 3.89% fraud rate)
        (F.when(F.col("price_out_of_range") == True, 60).otherwise(0)) +
        # is_blocklisted: real-world-sound signal, kept at low weight since it showed 0%
        # correlation in this dataset (documented limitation, not treated as strong evidence)
        (F.when(F.col("is_blocklisted") == True, 10).otherwise(0)) +
        # fx_rate_is_defaulted / currency_is_defaulted: data-quality flags, not fraud signals -
        # included at a small weight only to surface transactions with lower enrichment confidence
        (F.when(F.col("currency_is_defaulted") == True, 5).otherwise(0)) +
        (F.when(F.col("fx_rate_is_defaulted") == True, 5).otherwise(0))
    )
    .withColumn(
        "risk_tier",
        F.when(F.col("risk_score") >= 60, "high")
         .when(F.col("risk_score") >= 10, "medium")
         .otherwise("low")
    )
)

display(gold_risk_scored.groupBy("risk_tier").agg(
    F.count("*").alias("total_transactions"),
    F.sum("is_fraudulent").alias("fraud_count"),
    F.round(F.sum("is_fraudulent") / F.count("*") * 100, 2).alias("fraud_rate_pct")
).orderBy(F.desc("fraud_rate_pct")))

In [0]:
gold_risk_scored.write.mode("overwrite").saveAsTable("fraud_detection.gold.transactions_risk_scored")
display(spark.sql("SELECT COUNT(*) FROM fraud_detection.gold.transactions_risk_scored"))

In [0]:
daily_summary = (
    gold_risk_scored
    .withColumn("tx_date", F.to_date("transaction_date"))
    .groupBy("tx_date")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("is_fraudulent").alias("fraud_count"),
        F.round(F.sum("is_fraudulent") / F.count("*") * 100, 2).alias("fraud_rate_pct"),
        F.round(F.sum("transaction_amount_usd"), 2).alias("total_volume_usd")
    )
    .orderBy("tx_date")
)

display(daily_summary.limit(10))

daily_summary.write.mode("overwrite").saveAsTable("fraud_detection.gold.daily_fraud_summary")
display(spark.sql("SELECT COUNT(*) FROM fraud_detection.gold.daily_fraud_summary"))